In [1]:
print("Workspace initialized.")

Workspace initialized.


In [2]:
import kagglehub
import pandas as pd

# Download the latest version of the credit card fraud dataset
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
print("Path to dataset files:", path)

# Load the specific CSV file into a pandas DataFrame
# The download path contains 'creditcard.csv'
import os
csv_path = os.path.join(path, "creditcard.csv")
df = pd.read_csv(csv_path)

print("Dataset loaded successfully. Shape:", df.shape)

Using Colab cache for faster access to the 'creditcardfraud' dataset.
Path to dataset files: /kaggle/input/creditcardfraud
Dataset loaded successfully. Shape: (284807, 31)


In [3]:
# Print the exact count of transactions per class
print("Class Counts:")
print(df['Class'].value_counts())

# Print the percentage distribution
print("\nPercentage Distribution:")
print(df['Class'].value_counts(normalize=True) * 100)

Class Counts:
Class
0    284315
1       492
Name: count, dtype: int64

Percentage Distribution:
Class
0    99.827251
1     0.172749
Name: proportion, dtype: float64


In [4]:
from sklearn.model_selection import train_test_split

# Separate features (X) from the target label (y)
X = df.drop(columns=['Class'])
y = df['Class']

# Split into 80% training and 20% testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)
print("Fraudulent transactions in Training set:", y_train.sum())
print("Fraudulent transactions in Testing set:", y_test.sum())

Training features shape: (227845, 30)
Testing features shape: (56962, 30)
Fraudulent transactions in Training set: 394
Fraudulent transactions in Testing set: 98


In [5]:
from imblearn.over_sampling import SMOTE
from collections import Counter

# Initialize SMOTE
smote = SMOTE(random_state=42)

# Resample the training data only
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("Original training dataset class distribution:", Counter(y_train))
print("Resampled training dataset class distribution:", Counter(y_train_balanced))
print("\nNew shape of balanced training features:", X_train_balanced.shape)

Original training dataset class distribution: Counter({0: 227451, 1: 394})
Resampled training dataset class distribution: Counter({0: 227451, 1: 227451})

New shape of balanced training features: (454902, 30)


In [6]:
from sklearn.ensemble import RandomForestClassifier

# Initialize the model with controlled depth for speed and performance balance
model = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)

print("Training the Random Forest model on balanced data... (This can take 1-2 minutes)")
model.fit(X_train_balanced, y_train_balanced)
print("Model training complete.")

Training the Random Forest model on balanced data... (This can take 1-2 minutes)
Model training complete.


In [7]:
from sklearn.metrics import classification_report, confusion_matrix

# Generate predictions on the unseen test data
y_pred = model.predict(X_test)

# Print evaluation metrics
print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Legitimate (0)', 'Fraudulent (1)']))

--- Confusion Matrix ---
[[56806    58]
 [   12    86]]

--- Classification Report ---
                precision    recall  f1-score   support

Legitimate (0)       1.00      1.00      1.00     56864
Fraudulent (1)       0.60      0.88      0.71        98

      accuracy                           1.00     56962
     macro avg       0.80      0.94      0.86     56962
  weighted avg       1.00      1.00      1.00     56962



In [8]:
import joblib
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel
import numpy as np
import time

# 1. Serialize (save) the trained model to disk
joblib.dump(model, 'fraud_model.pkl')
print("Model serialized and saved as 'fraud_model.pkl'")

# 2. Initialize FastAPI application infrastructure
app = FastAPI()

# Load the model into the API lifecycle
loaded_model = joblib.load('fraud_model.pkl')

# Define data structure validation for incoming JSON payloads
class TransactionData(BaseModel):
    features: list

# Define the low-latency prediction endpoint
@app.post("/predict")
def predict_fraud(data: TransactionData):
    start_time = time.time()

    # Reshape flat list input into 2D array expected by the model
    input_matrix = np.array(data.features).reshape(1, -1)

    # Execute inference
    prediction = loaded_model.predict(input_matrix)
    probability = loaded_model.predict_proba(input_matrix)[0][1]

    latency_ms = (time.time() - start_time) * 1000

    return {
        "is_fraud": int(prediction[0]),
        "fraud_probability": round(float(probability), 4),
        "inference_latency_ms": round(latency_ms, 2)
    }

# 3. Simulate an production environment API call inside Colab using TestClient
client = TestClient(app)

# Extract a single sample transaction row from your test set (Legitimate sample)
sample_transaction = X_test.iloc[0].tolist()

# Send a mock POST request to the API endpoint
print("\nSending mock live transaction data to /predict endpoint...")
response = client.post("/predict", json={"features": sample_transaction})

# Output production metrics
print("Production API API Response:")
print(response.json())

Model serialized and saved as 'fraud_model.pkl'

Sending mock live transaction data to /predict endpoint...
Production API API Response:
{'is_fraud': 0, 'fraud_probability': 0.003, 'inference_latency_ms': 122.88}


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
